### Identify next batch id to process and set to Job's task values

In [0]:
%run ../0-common/env-config

In [0]:
control_table = f"{catalog_name}.{control_schema}.batch_control"

In [0]:
from pyspark.sql import functions as F

# Get batch folders from landing
landing_batches = sorted([
    file.name.rstrip("/")
    for file in dbutils.fs.ls(landing_folfer_path)
    if file.isDir()
])

# Read tracked batches
if spark.catalog.tableExists(control_table):
    tracked_batches = [
        row.batch_id
        for row in (
            spark.table(control_table)
            .filter(F.col("status").isin("in_progress","completed"))
            .select("batch_id")
            .distinct()
            .collect()
        )
    ]
else:
    tracked_batches = []

# Identify earliest unprocessed
new_batches = sorted(list(set(landing_batches)-set(tracked_batches)))
next_batch = new_batches[0] if new_batches else None

print(F"Landing batched        :{landing_batches}")
print(F"Tracked batched        :{tracked_batches}")
print(F"Next batch to process  :{landing_batches}")

if next_batch is None:
    dbutils.jobs.taskValues.set(key="batch_id",value="")
    dbutils.jobs.taskValues.set(key="has_batch",value="false")
else:
    dbutils.jobs.taskValues.set(key="batch_id",value=next_batch)
    dbutils.jobs.taskValues.set(key="has_batch",value="true")